# Hello all, I hope you'll find this notebook interesting. With many public notebooks out there, this notebook is **the highest scoring** using logistic regression with CV:0.95550

Let's get started

# Import

I earlier used cross_val_score for cross-validation but then moved to KFold so that OOF predictions can be saved.

In [1]:
# imports
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.linear_model import LogisticRegression # Model
from sklearn.model_selection import cross_val_score, KFold # Model Evaluation
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore") # ignore, lol

# Load

The column ```id``` is dropped from the dataset as it does not hold any strong signal, but since it is one of the important value needed for submission of predictions, ```test.id``` is preserved.

In [2]:
# Loading Data
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv') 
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')

# Setting the target - category to 0/1
train['Heart Disease']=train['Heart Disease'].map({'Absence':0, 'Presence':1})
train.drop('id', axis=1, inplace=True) # id is not used for model training and prediction
testID=test.pop('id') # important to preserve test IDs for final submission.
y=train.pop('Heart Disease') # TARGET

train.head(3)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3


Below we can see the number of unique values in the data

In [3]:
train.nunique()

Age                         42
Sex                          2
Chest pain type              4
BP                          66
Cholesterol                150
FBS over 120                 2
EKG results                  3
Max HR                      93
Exercise angina              2
ST depression               66
Slope of ST                  3
Number of vessels fluro      4
Thallium                     3
dtype: int64

Ideally we one-hot encode features with low cardinality, but since the cardinality is not high compared to the amount of data, we can try encoding all. This would generate around 430 features

In [4]:
BASE_FEATURES=list(train.columns)

# ONE-HOT

In [5]:
# One Hot All columns 
# (Selective Encoding can be done, but that does not help here)
oh_cols = BASE_FEATURES
oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

oh_train = pd.DataFrame(oh.fit_transform(train[oh_cols]))
train[oh.get_feature_names_out(train[oh_cols].columns)] = oh_train.astype(int) #.astype('category')

oh_test= pd.DataFrame(oh.transform(test[oh_cols]))
test[oh.get_feature_names_out(test[oh_cols].columns)] = oh_test.astype(int) #.astype('category')

Why not One-Hot encode only the categorical features. 

But take a close look on the data, **630K** records. And the max unique values in any column is note more than 150. Perform EDA if you wish, but let me put some real stats, not one-hot encoding features that seem numerical performs led to a poor CV and LB.

**Why does it work?**

Data is huge. Cardinality of each column is very very low compared to the size of data. Though features may be continuous (numeric), still they repeat so many times.

* Detailed explaination is in the comments, check that.

In [6]:
# Target Encoding
train['y']=y

TE=[]
for col in BASE_FEATURES:
    temp = train.groupby(col).y.agg('median') # Median would be same as mode for this case
    train[f'{col}_mode'] = train[col].map(temp)
    test[f'{col}_mode'] = test[col].map(temp)
    TE.append(f'{col}_mode')

    temp = train.groupby(col).y.agg('mean') # Call it simple probability
    train[f'{col}_prob'] = train[col].map(temp)
    test[f'{col}_prob'] = test[col].map(temp)
    TE.append(f'{col}_prob')

train.pop('y')
print("Target Encoded")

Target Encoded


Target encoding is usually recommended to be performed inside the cross validation folds to avoid leakage. But I don't think there is going to be some serious leakage in the code above, if it happens, public LB shall warn us.

In [7]:
test = test.fillna(0.5) # target encoding led to some NaN values

# Scaling Data

This is an important step because we are using Logistic Regression.

Two benefits are : 

1. Better performance
2. Becomes Faster

In [8]:
# Speed of Logistic Regression Increases 
# (results shall also increase, but speed is more observable)
scaler = StandardScaler()
scaled_train = pd.DataFrame(scaler.fit_transform(train))
scaled_train.columns = train.columns
scaled_test = pd.DataFrame(scaler.transform(test))
scaled_test.columns = test.columns

In [9]:
scaled_train.head()

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,...,Exercise angina_mode,Exercise angina_prob,ST depression_mode,ST depression_prob,Slope of ST_mode,Slope of ST_prob,Number of vessels fluro_mode,Number of vessels fluro_prob,Thallium_mode,Thallium_prob
0,0.467921,0.631760,0.806994,1.435822,-0.178490,-0.294858,-0.982858,0.271190,1.628894,3.040655,...,1.628894,1.628894,1.339894,1.997396,1.148335,1.14035,1.556068,1.947827,1.201903,1.218284
1,-0.258797,0.631760,-2.715729,-0.367088,2.374837,-0.294858,1.019582,0.951359,-0.613913,-0.754928,...,-0.613913,-0.613913,-0.746328,-0.801572,-0.870826,-0.87064,-0.642646,-0.630300,-0.832014,-0.830742
2,0.225682,-1.582881,-1.541488,1.970017,-1.692672,-0.294858,1.019582,-0.095054,-0.613913,-0.754928,...,-0.613913,-0.613913,-0.746328,-0.801572,-0.870826,-0.87064,-0.642646,-0.630300,-0.832014,-0.830742
3,-1.227755,-1.582881,-0.367247,0.233882,-0.475388,-0.294858,1.019582,-0.147375,-0.613913,0.299400,...,-0.613913,-0.613913,1.339894,0.596236,1.148335,1.14035,-0.642646,-0.630300,-0.832014,-0.830742
4,0.467921,0.631760,0.806994,0.634529,-0.326939,-0.294858,1.019582,-1.455391,1.628894,3.251520,...,1.628894,1.628894,1.339894,1.951692,1.148335,1.14035,1.556068,1.958557,-0.832014,-0.830742


# MODEL

Increasing the max iterations could be worth trying because the model does not converge with default parameters.

In [10]:
model = LogisticRegression() # Our Model, simple one - Logistic Regression

In [11]:
kf = KFold(n_splits=5, shuffle=False)
i=0
oof=pd.Series(index=scaled_train.index)
test_p=[]
for train_index, val_index in kf.split(scaled_train):
    i+=1
    X_t, y_t = scaled_train.iloc[train_index], y.iloc[train_index]
    X_val, y_val = scaled_train.iloc[val_index], y.iloc[val_index]
    model.fit(X_t, y_t)
    
    temp=model.predict_proba(X_val)[:,-1]
    score = roc_auc_score(y_val, temp)
    print(f'Fold {i} \n Score: {score}')
    oof.iloc[val_index] = temp
    
    pred=model.predict_proba(scaled_test)[:,-1]
    test_p.append(pred)

print(f"CV: {roc_auc_score(y, oof)}")

Fold 1 
 Score: 0.9559825015926918
Fold 2 
 Score: 0.9551701629072537
Fold 3 
 Score: 0.9560240029050451
Fold 4 
 Score: 0.9548873308820172
Fold 5 
 Score: 0.9554444276020823
CV: 0.9555025172610422


* CV improvement on sixth decimal place, and maybe LB had also improvement over sixth decimal places only, which is a reason why no changes are visible 😂

* So, maybe we need a better way to target encode or Logistic Regression has reached it's upper limit.

# Final Submission

In [12]:
model.fit(scaled_train, y)

LogisticRegression()

In [13]:
out=model.predict_proba(scaled_test)[:,-1]

final = pd.DataFrame({'id':testID, 'Heart Disease':out})
final.to_csv('submission.csv', index=False) # submission file'
print("Submitted Succesfully!!")
final.head()

Submitted Succesfully!!


,id,Heart Disease
0,630000,0.942355
1,630001,0.009023
2,630002,0.992445
3,630003,0.004612
4,630004,0.167251


# Saving OOF predictions

In [14]:
oof.to_csv('oof_lr.csv', index=False)

# Next Steps

1. Model coefficients and other metrics like mutual info can be used to find the important features and the non-contributing ones can be dropped.
2. New features can be generated using interactions of two or more features, or target encoding can also be performed.
3. Instead of one-hot encoding all, using the first point, selective encoding can be performed.

# This notebook helped? Did it provide some value?

**If yes, consider giving an upvote**

*Have any doubts or suggestions - Do comment*